In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("adh_ai_labs") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "adh-ai-labs @ git+https://github.com/BradSegal/OxfordADH2026AILabs.git"], check=True)
    importlib.invalidate_caches()
print("Course helpers ready.")

# Introduction to deep learning: detecting atrial fibrillation from ECGs

**MSc in Applied Digital Health.** About 60 minutes.

You were introduced to the basics of neural networks this morning. Now let's put this into practice by training two models on real-world ECG data to answer the question: **does this 10-second ECG show atrial fibrillation (AF)?**

| Part | Time | What you do |
|---|---|---|
| Set-up | 8 min | Nature of data (what an ECG records, what AF looks like, how an ECG becomes numbers) |
| 1 | 20 min | Train a model on two rhythm features and visualise how it learns |
| 2 | 8 min | Fairly evaluate the model on unseen patients |
| 3 | 19 min | Train a CNN on raw ECG and spot overfitting |
| Wrap-up | 4 min | A review of what a deep neural network learns and where it leads |

### How this notebook works

- Each cell can be run in sequence by clicking inside it and pressing **Shift + Enter**. If you experience any errors, you can go to *Runtime > Restart session and run all*.
- Each **TODO** cell requires 1-3 lines of code to complete. A code check is provided at each cell that will print ✅ if your code is correct, ❌ with a hint if it isn't, or ⏳ if the placeholder code has not yet been replaced.
- **Check your understanding:** type your answer within the quotation marks, e.g., `check("4a", "c")`, and run the cell.
- The **extensions** at the end are for if you have extra time.

### Before you start

1. **Data:** Download the lab data file `adh_multimodal_lab_data.zip` from Canvas and upload the zip file to `MyDrive/adh_multimodal_lab/adh_multimodal_lab_data.zip` on Google Drive. Link your Drive when requested.
2. **GPU (optional):** *Runtime > Change runtime type > T4 GPU*. If this is not enabled then the CNN will take a minute to train instead of a few seconds.
3. You must save your own copy before editing. *File > Save a copy in Drive*.

The ECGs and automated machine reports are real, from the open MIMIC-IV-ECG database (ODbL licence); the included health records are simulated and are not used in this lab. Please ensure you keep the data within the course and do not paste any of the data into an AI chatbot.

## Set-up

The next cell sets up the libraries that this lab will use, just as you did in the bootcamp when you imported NumPy with `import numpy as np`. The main library we will be using is **PyTorch** (`torch`), a library for building and training neural networks, and we will use `adh_ai_labs` for the visualisations, diagrams and checks for this course. This code is hidden as you do not need to look at it.

In [ ]:
# @title Load the libraries { display-mode: "form" }
import copy
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

from adh_ai_labs import diagrams, ecg
from adh_ai_labs.checks import intro_to_deep_learning as checks
from adh_ai_labs.checks.intro_to_deep_learning import check
from adh_ai_labs.data import load_ecgs

SEED = 0  # fixes the random choices, so everyone gets the same results
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", "a GPU" if device == "cuda" else "the CPU (no GPU): that's fine, Part 3 just takes longer")

### Where the lab is going

In [ ]:
diagrams.roadmap()

### The data

`load_ecgs` unzips the data and returns two items, which the next line of code splits into two variables simultaneously: `meta`, which is a table of the patient ID, age, sex, and the machine report for each ECG, and `ecgs`, the actual ECG data.

In [ ]:
meta, ecgs = load_ecgs()

### What an ECG records

The ECG records electrical activity of the heart from multiple directions. Each direction is recorded as a lead. Lead II is used throughout this lab and produces a clear view of each cardiac cycle. The next cell visualises a single cardiac cycle and annotates the various components.

In [ ]:
sinus_example = 226  # an ECG in normal sinus rhythm
diagrams.annotated_beat(ecgs, sinus_example)

### What AF looks like

Atrial fibrillation (AF) is a condition whereby the top chambers of the heart (atria) fibrillate instead of normally contracting. This results in a lack of P waves and disordered beat intervals. This is a concerning condition as it is the most common form of sustained arrhythmia and is a risk factor for stroke. Given that the condition can be paroxysmal (intermittent), it can be missed on an isolated ECG. A comparison of ECGs is demonstrated below. Note the differences in P waves and the beat intervals.

In [ ]:
af_example = 75  # an ECG whose report says atrial fibrillation
diagrams.sinus_vs_af(ecgs, sinus_example, af_example)

### The label

The label for each ECG will be whether it is AF (1) or not (0). This will be obtained from the machine report, where a label of 1 will be given if it describes the ECG as having atrial fibrillation. This is not a cardiologist reading and so will have some erroneous labels; however, this is typically the case with real-world data.

In the next cell, `meta["machine_report"]` selects the reports column (similar to a dictionary key), `.str.contains(...)` checks if the text contains the words "atrial fibrillation", and `.to_numpy().astype(int)` converts the result to an array of 1s and 0s.

In [ ]:
af = meta["machine_report"].str.contains("atrial fibrillation", case=False).to_numpy().astype(int)
print(f"{af.sum()} of {len(af)} ECGs mention atrial fibrillation")

### An ECG as numbers

To a computer, an ECG is a list of numbers for each lead, similar to how the X-rays were a grid of numbers in the bootcamp but with an added dimension. The diagram zooms in on a single trace and then shows how the entire dataset is stored. The cell then prints the shape of the array.

In [ ]:
diagrams.ecg_as_numbers(ecgs, sinus_example)
print("ecgs.shape:", ecgs.shape)

## Part 1: a model is a function with adjustable numbers (20 min)

### Two numbers a clinician would use

The first two pieces of information a clinician evaluating a rhythm strip would look at are the rate and rhythm regularity, which `ecg.rhythm_features` determines from the lead II trace as follows:

- **Heart rate** (BPM): derived from the median beat-to-beat interval.
- **RR irregularity**: beat-to-beat variation, quantified as the standard deviation of the RR interval lengths divided by the mean. A perfectly regular rhythm will have a value of 0.

These **features** are numbers that we extract from the raw data that we think may be useful. This is leveraging clinical knowledge that the CNN in Part 3 does not have access to. The next cell calculates both of these features for all 1000 ECGs and plots a single point for each. See if you can see where the red (AF) dots are in relation to the blue ones.

In [ ]:
features = ecg.rhythm_features(ecgs)
ecg.plot_features(features, af)

Note the overall irregular pattern and increased heart rate of most AF ECGs (red); however, there is substantial overlap without an obvious cut-off. The model must learn how to partition this from data instead of by inspection.

### Putting some patients aside first

Prior to any training, we split off 20% of the **patients** to form a **test set**. We do not show these patients to the model during training and use them in Part 2 to determine how well the model performs on unseen patients.

`GroupShuffleSplit`, from scikit-learn, splits the patients randomly but keeps all ECGs for a particular patient together. This can produce several splits and so `next(...)` selects the first one. This returns two arrays of indices: `train_idx` is the indices of the training ECGs and `test_idx` is the indices of the test ECGs. The cell then prints out how many ECGs there are in each group as well as the number that have AF.

In [ ]:
split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(split.split(ecgs, groups=meta["subject_id"]))
print(f"Training: {len(train_idx)} ECGs, {af[train_idx].sum()} with AF")
print(f"Test:     {len(test_idx)} ECGs, {af[test_idx].sum()} with AF")

`af[train_idx]` will extract the labels for these positions. Arrays can be indexed by a list of positions as well as by a slice such as `[2:5]`.

### Putting the two features on the same scale

Models learn much more effectively if their inputs are similarly **scaled**. Our heart rates have an average of around 80 while the irregularity measure is about 0.05. We want to normalise these features by subtracting the average and dividing by the standard deviation (a measure of how spread out a set of values are). This is calculated only on the training ECGs to prevent information leaking from the test patients. A value of 0 represents the average, and 1 is one standard deviation above the average. `features.iloc[train_idx]` selects the training rows of our table based on the row number.

In [ ]:
mean = features.iloc[train_idx].mean()
std = features.iloc[train_idx].std()
scaled = ((features - mean) / std).to_numpy()
print("Scaled features of the first three ECGs:")
print(scaled[:3].round(2))

### From NumPy arrays to PyTorch tensors

PyTorch leverages **tensors** instead of NumPy arrays. Tensors are PyTorch's implementation of an array; they have similar properties in that they contain numbers in tabular form and can be sliced similarly. Tensors have an additional property in that they can track the history of operations they are involved in for PyTorch to leverage for training. `torch.tensor(...)` converts the array to a tensor, `dtype=torch.float32` casts the values to decimal format, and `.unsqueeze(1)` reshapes the 1000 labels into a single column (1000x1) to match the model's output.

In [ ]:
x = torch.tensor(scaled, dtype=torch.float32)  # one row per ECG, two columns
y = torch.tensor(af, dtype=torch.float32).unsqueeze(1)  # one column: 1 for AF, 0 for not
x_train, y_train = x[train_idx], y[train_idx]
print("x_train:", tuple(x_train.shape), " y_train:", tuple(y_train.shape))

### The model

The model takes each scaled feature and multiplies it by a **weight**. These are then summed alongside a **bias** term. The final **score** is then squashed between zero and one by means of a **sigmoid** function to produce a probability.

In [ ]:
diagrams.model_diagram()

The same model as formulas, one step at a time. For an individual ECG, we write $x_1$ for the scaled heart rate and $x_2$ for the scaled RR irregularity value. The model itself has three numbers, called **parameters**: the weights $w_1$ and $w_2$, which say how important each of the features is, and the bias $b$ which is where the model starts from. The **score** is given by

$$z = w_1 x_1 + w_2 x_2 + b$$

This score can be any real number and so is passed through the **sigmoid** function $\sigma$ to transform it to a probability $p$ between zero and one:

$$p = \sigma(z) = \frac{1}{1 + e^{-z}}$$

If $z$ equals 0, $e^{0} = 1$ and $p = 1/2$. If $z$ is large positive, $e^{-z}$ will be very small and $p$ will be close to 1. If $z$ is large negative, $e^{-z}$ will be very large and $p$ will be close to 0. This model is known as **logistic regression**.

In PyTorch, the entire model is merely a single line, `nn.Linear(2, 1)`. This line represents a function that takes in 2 numbers and produces a score. `model` is an **object** that contains a combination of numbers and functions, similar to `ecgs`, which has an associated shape. `model.weight` stores $w_1$ and $w_2$ while `model.bias` stores $b$. These are initialised to small random values by PyTorch and so the model is effectively guessing at the moment.

In [ ]:
torch.manual_seed(SEED)
model = nn.Linear(2, 1)
print("w1, w2:", model.weight.data)
print("b:     ", model.bias.data)

Let's go through the maths on the AF ECG with these random parameters by hand. `.item()` converts a tensor with a single value to a standard Python number. The final line uses the model itself, i.e. `torch.sigmoid(model(...))`; the two probabilities should match.

In [ ]:
x1, x2 = x[af_example, 0].item(), x[af_example, 1].item()
w1, w2, b = model.weight[0, 0].item(), model.weight[0, 1].item(), model.bias[0].item()
z = w1 * x1 + w2 * x2 + b
p = 1 / (1 + np.exp(-z))
print(f"ECG {af_example}: x1 = {x1:.2f}, x2 = {x2:.2f}")
print(f"z = {w1:.2f} × {x1:.2f} + {w2:.2f} × {x2:.2f} + ({b:.2f}) = {z:.2f}")
print(f"p = 1 / (1 + e^(-z)) = {p:.2f}")
print(f"The model's own answer: {torch.sigmoid(model(x[af_example])).item():.2f}")

### How wrong is it? The loss

To get better, we need a single value that tells us how wrong we are: the **loss** value. For a yes / no question the standard loss is the **binary cross-entropy**. For a single ECG with label $y$ (1 for AF, 0 for not) and predicted probability $p$:

$$L = -\big[\, y \log p + (1 - y) \log(1 - p) \,\big]$$

Remember there's only one of the two terms for a given ECG because $y$ is either 1 or 0:

- if the ECG is AF ($y = 1$), the second term is multiplied by 0 and $L = -\log p$;
- if it does not ($y = 0$), the first term is multiplied by 0 and $L = -\log(1 - p)$.

($\log$ denotes the natural log.) For $N$ ECGs the loss is averaged, $\frac{1}{N}\sum_{i=1}^{N} L_i$. This can be seen graphically as $L$ for both cases, with a confident correct answer having minimal loss and a confident incorrect prediction having a substantial loss.

In [ ]:
diagrams.loss_curves()

The next cell computes $L$ for the AF ECG ($y = 1$) with the calculated probability as well as the sinus rhythm ECG from the set-up ($y = 0$). It then computes the average loss for all $N$ training ECGs using the PyTorch `nn.BCEWithLogitsLoss` function. This function takes the scores (not the probabilities) and computes the sigmoid internally, which is more accurate.

In [ ]:
print(f"ECG {af_example} has AF (y = 1): L = -log({p:.2f}) = {-np.log(p):.2f}")
p_sinus = torch.sigmoid(model(x[sinus_example])).item()
print(f"ECG {sinus_example} has no AF (y = 0): p = {p_sinus:.2f}, L = -log(1 - {p_sinus:.2f}) = {-np.log(1 - p_sinus):.2f}")
loss_fn = nn.BCEWithLogitsLoss()
print(f"Mean loss over the N = {len(y_train)} training ECGs: {loss_fn(model(x_train), y_train).item():.3f}")

### Going downhill: gradient descent

The goal of training is to find a way to modify the values of $w_1$, $w_2$ and $b$ to minimise the loss.
The **gradient** of the loss for each parameter indicates the slope of the function, and thus, how much the loss would change if the parameter was increased slightly.
**Gradient descent** is a method of iteratively stepping the parameters in the opposite direction to the slope (downhill) by some factor (the **learning rate**) multiplied by the slope.
This can be visualised for a single parameter $w_2$ on the actual loss landscape for our model below for different learning rates:

In [ ]:
diagrams.gradient_descent_1d(x_train, y_train, [0.48, 0.98], -2.58)

A good learning rate will get to the bottom in a few steps; a bad learning rate will crawl or bounce around the valley. For all three parameters, the idea is the same. PyTorch will determine the slope for each parameter and an **optimiser** will take a step. Every neural network is trained by repeating this loop:

In [ ]:
diagrams.training_loop()

### TODO 1: one training step

`train_step` is a function (as in Level 6 of the bootcamp) for a single loop iteration. It takes as inputs the model, the optimiser, the ECGs `x`, and their labels `y`. Replace the placeholder line with 3 lines of code, one for each step:

1. **Forward pass & loss** (1 & 2): Score `x` with `model` and calculate the loss with `loss_fn`, passing the scores then `y` as the labels (in that order). Finally, save the output as `loss`.
2. **Backward pass** (3): Call the `.backward()` method on the `loss` object. A method is a function that is defined on an object, such as `.append()` for a list. This method calculates the slope for all parameters.
3. **Step** (4): Call the `.step()` method of `optimiser`. This will move all parameters.

The first line of code, `optimiser.zero_grad()`, resets the slope. This is necessary because, by default, PyTorch will keep adding each new slope to the previous one. The last line of code, `return loss.item()`, returns the loss as a numeric value. We will use this function later to train the CNN in Part 3.

In [ ]:
def train_step(model, optimiser, x, y):
    """One step of gradient descent on inputs x with labels y. Returns the loss before the step."""
    optimiser.zero_grad()  # clear the gradients from the last step
    loss = torch.tensor(float("nan"))  # placeholder: replace this line with the three steps
    return loss.item()


checks.train_step(train_step)

### Watch it learn

Below we create a new model and optimiser, `torch.optim.SGD`, with a learning rate of 0.5. In Part 1, we will use all training ECGs in each iteration, so this is vanilla gradient descent. The `for` loop runs `train_step` 100 times, and `step % 5 == 0` (if the step is a multiple of 5) redraws the figure. Observe on the left how the black line (the **decision boundary**, where the AF prediction probability is 0.5) gradually improves the separation of the data; on the right, you can see the loss progressively decreasing.

In [ ]:
torch.manual_seed(SEED)
model = nn.Linear(2, 1)
optimiser = torch.optim.SGD(model.parameters(), lr=0.5)
picture = ecg.TrainingPlot(features.iloc[train_idx], af[train_idx], mean, std, model)

losses = []
for step in range(1, 101):
    losses.append(train_step(model, optimiser, x_train, y_train))
    if step % 5 == 0:
        picture.update(model, losses)

The loss quickly comes down before plateauing; it has **converged**. The diagram below demonstrates the journey the model's two weights take as they jostle their way down the loss landscape (each point represents 5 steps).

In [ ]:
diagrams.loss_landscape(x_train, y_train, picture.path, model.bias.item())

Finally, the trained model with its learnt numbers, worked through for the same AF ECG as before. Compare its probability with the untrained model's.

In [ ]:
diagrams.model_diagram(model, x[af_example].numpy(), f"ECG {af_example} (labelled AF)")

Given the same scale, we can even compare the weights. The model has learnt from the data what a clinician already knows. It has determined that a single standard deviation increase in irregularity is about twice as important as a standard deviation increase in heart rate for the final score.

## Part 2: testing it fairly (8 min)

### Why keep whole patients aside?

The model will generally perform well on ECGs it has seen before, but this does not represent new patients. We split by **patients** and not ECGs as some patients have multiple ECGs (1000 ECGs but only 985 patients) and these ECGs are generally quite similar. This means the model could perform well just by knowing which patient it is looking at. This next cell is just to check that there are no patients that are in both groups. `set(...)` ensures we just have unique patient IDs, `&` ensures we only have patient IDs in both groups. We're hoping to see 0.

In [ ]:
shared = set(meta["subject_id"].iloc[train_idx]) & set(meta["subject_id"].iloc[test_idx])
print("Patients in both the training and the test set:", len(shared))

### Scoring the test patients without changing the model

During training, PyTorch keeps track of all computations performed in a model, which `loss.backward()` uses to determine how to update the model to improve performance. If we're only interested in the predictions of the model, this is wasteful as we require additional memory and processing. This can be switched off by use of `torch.no_grad()`, which disables this tracking for any indented lines following the statement `with`.

In [ ]:
diagrams.no_grad_diagram()

`predict` performs two independent steps prior to scoring the model. `model.eval()` indicates to the model that it is in evaluation mode. This is important as several layers, such as the BatchNorm layers in the CNN in Part 3, have different behaviours for training and prediction. This does not disable tracking. `with torch.no_grad():` disables tracking. `.squeeze(1)` collapses the column of scores to a flat list. `.cpu().numpy()` converts the tensor to a NumPy array. A histogram visual of the test ECG scores is produced. Note how much overlap there is between the red (AF) and blue groups.

In [ ]:
def predict(model, x):
    """The model's score for each row of x, as a NumPy array."""
    model.eval()
    with torch.no_grad():
        return model(x).squeeze(1).cpu().numpy()


x_test, af_test = x[test_idx], af[test_idx]
test_scores = predict(model, x_test)
ecg.plot_scores(test_scores, af_test, "Two-feature model: scores for the test patients")

Most of the AF ECGs score higher than most of the others, but the two groups overlap. For any score cut-off, we would miss some AF cases and have some false alarms.

### AUROC: one number for how well the scores separate

The area under the receiver operating characteristic curve (**AUROC**) is a single number quantification of a model's score separation performance. It indicates the probability that the model will score a random AF ECG higher than a random non-AF ECG. The metric is insensitive to score cut-off and is 0.5 for no better than random and 1 for perfect classification performance. This metric is computed using the scikit-learn function `roc_auc_score(labels, scores)` by providing the true labels followed by the model's scores.

### TODO 2: the test AUROC

Set `linear_auroc` to the AUROC for `model` on the test patients with test labels `af_test` and test scores `test_scores`.

In [ ]:
linear_auroc = ...
checks.test_auroc(linear_auroc, model)

**Check your understanding**

**2a.** The test AUROC is about 0.89.
What do you think this means?

- a) The model is able to correctly label 89% of the test ECGs.
- b) The model is able to detect 89% of AF ECGs.
- c) If you took a random test ECG with AF and another without, the one with AF would have a higher score 89% of the time.
- d) The model is 89% certain of the result for each ECG.

In [ ]:
check("2a", "")

## Part 3: letting the network find its own features (19 min)

The model from Part 1 only had access to the two features that we gave it. A **Convolutional Neural Network** (CNN) takes the raw signal, all 2500 numbers of lead II, and learns what features it needs to look for. The core building block of this is the **convolution**, and we build up to this in four steps.

### Step 1: a convolution by hand

Let's take a brief scenario where we have a short list of numbers (our **signal**) and a smaller list of numbers (our **filter**). We place the filter over the start of our signal and then multiply the numbers that line up and add the products to get our first result. We then move the filter one step to the right and repeat the process until we've gone through the entire signal. Below is our signal of 8 numbers, which shows a small beat, and our filter [-1, 0, 1]. You can verify the first row or two by yourself.

In [ ]:
diagrams.convolution_by_hand([0, 0, 1, 3, 2, 1, 0, 0], [-1, 0, 1])

The output is positive when the signal is increasing (the filter takes the left number and subtracts it from the right number), negative when the signal is decreasing and 0 when the signal is flat. The subsequent cell executes the same process in Python using lists, slices, and `for` loops as you've seen in the bootcamp. You should arrive at the same 6 numbers.

In [ ]:
signal = [0, 0, 1, 3, 2, 1, 0, 0]
kernel = [-1, 0, 1]
output = []
for start in range(len(signal) - 2):
    window = signal[start:start + 3]  # the three numbers under the filter
    output.append(window[0] * kernel[0] + window[1] * kernel[1] + window[2] * kernel[2])
print(output)

Try changing `kernel` to `[1/3, 1/3, 1/3]` and re-run the cell.

### Step 2: the same idea as the X-rays

With `[1/3, 1/3, 1/3]`, each output is the average of the three numbers under the filter. It is the same as performing `xray[100:300, 50:450].mean()` in the bootcamp but on a small window and moving one step at a time. A different filter looks for different patterns: [-1, 0, 1] looks for rises, and the average smooths. CNNs for images work the same way but with a small square filter moving over pixels in a grid. Here it moves in only a single dimension over time.

### Step 3: on a real ECG beat

Here is the same filter, [-1, 0, 1], at 3 positions on a real beat of lead II, constructing the output as it goes:

In [ ]:
diagrams.convolution_steps(ecgs[sinus_example, 1], [-1, 0, 1])

The output will be high at the steep upstroke of the QRS and strongly negative on the downslope.

### Step 4: letting the network choose the filters

We selected the filter [-1, 0, 1] because we knew what we wanted to find. The first layer of a CNN is a set of filters similar to this; however, the numbers start out random and are learnt via gradient descent in the same manner as $w_1$, $w_2$ and $b$ from Part 1. The next layer has filters that look at the outputs of the first layer and so can learn patterns of patterns without needing to know beforehand what to look for.

### The CNN

We construct our CNN by stacking four **convolutional blocks**. Each block contains a convolutional layer `nn.Conv1d` that convolves a set of learned filters, each nine samples (36 ms) long, across the signal, a batch normalisation layer `nn.BatchNorm1d` that ensures the outputs from the convolution are in a reasonable range, a rectified linear unit `nn.ReLU` that forces negative values to zero, enabling the network to learn non-linear patterns, and a max pooling layer `nn.MaxPool1d(4)` that takes the maximum value of every four samples. This is then followed by a global max pooling layer `nn.AdaptiveMaxPool1d(1)` that takes the maximum value of each channel across the entire signal length. This produces a 64-dimensional **embedding** that summarises the ECG signal.

A **channel** here is one time series (i.e., lead II starts as a single channel; this becomes 16 channels after the first block using 16 filters). A filter uses all input channels and looks at a segment of time (nine samples here) to produce an output channel. `padding=4` pads the beginning and end of the sequence with four zeroes each to ensure the filter outputs a sequence of equivalent length. `nn.MaxPool1d(4)` is a downsampling layer that divides the length of the sequence by four (nearest integer down).

In [ ]:
diagrams.cnn_diagram()

The use of blocks reduces the signal length by a factor of four each time, which means that features later in the network summarise larger regions of the ECG up to just under four seconds in length. The final layer, `nn.Linear(64, 1)`, is the Part 1 model except with 64 learned features instead of 2 hand-constructed features.

In the code below, `nn.Sequential(...)` sets the model operation order, `conv_block` is a block, and `make_cnn` is the four stacked blocks. `ecgs[:, 1:2]` takes all ECGs (`:`) but only lead II (`1:2` keeps the lead dimension). The last line is a parameter count.

In [ ]:
def conv_block(in_channels, out_channels):
    """One convolutional block: learnt filters, rescaling, ReLU, then pooling."""
    return nn.Sequential(
        nn.Conv1d(in_channels, out_channels, kernel_size=9, padding=4),
        nn.BatchNorm1d(out_channels),
        nn.ReLU(),
        nn.MaxPool1d(4),
    )


def make_cnn(in_channels=1):
    """A small CNN for 10-second ECGs; in_channels is the number of leads."""
    return nn.Sequential(
        conv_block(in_channels, 16),
        conv_block(16, 32),
        conv_block(32, 64),
        conv_block(64, 64),
        nn.AdaptiveMaxPool1d(1),  # the largest value of each channel over the whole recording
        nn.Flatten(),  # 64 numbers per ECG: the embedding
        nn.Linear(64, 1),  # one score per ECG, as in Part 1
    )


torch.manual_seed(SEED)
cnn = make_cnn()
signals = torch.tensor(ecgs[:, 1:2])  # lead II only, shaped (ECGs, 1 channel, 2,500 time points)

n_parameters = sum(p.numel() for p in cnn.parameters())  # add up the size of every weight tensor
print(f"Parameters to learn: {n_parameters:,} (the Part 1 model had 3)")

### A third group: validation patients

The CNN has about 60,000 parameters but very few ECGs to train on; as a result, it is able to memorise the training set, including the noise and incorrect labels, which will result in poor performance on new patients. This is known as **overfitting**. To detect this without accessing the test set, we split the training patient population a second time into a training (80%) and **validation** (20%) set, which is evaluated after each time the model has seen all the training data. This cell leverages the same tool as before and will output the size of all three sets.

In [ ]:
inner = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
learn, check_on = next(inner.split(train_idx, groups=meta["subject_id"].iloc[train_idx]))
cnn_train_idx, val_idx = train_idx[learn], train_idx[check_on]
print(f"CNN training: {len(cnn_train_idx)} ECGs, validation: {len(val_idx)} ECGs, test: {len(test_idx)} ECGs")

### Mini-batches: stochastic gradient descent

In Part 1 every step used all the training ECGs. With big data that is slow, so neural networks usually take each step using a **mini-batch**: a few ECGs picked at random. Each mini-batch gives a slightly different slope, so the path is noisier, but each step is far cheaper. This is **stochastic gradient descent** (stochastic means random). The figure compares the two on the Part 1 landscape:

In [ ]:
diagrams.full_batch_vs_mini_batch(x_train, y_train, model.bias.item())

### The training loop

`fit` trains the CNN using your `train_step` from TODO 1 in mini-batches of 32 ECGs. A single pass through all the training ECGs (~20 mini-batches of 32 ECGs each) is called an **epoch**. The CNN is trained using the **Adam** optimiser, which is a variation of SGD that scales the step size for each parameter dynamically. This has become the default optimiser for most deep learning tasks. `fit` is the logic loop that handles the optimisation of the network, calculating and logging the loss for the training and validation sets every epoch as well as saving a copy of the model weights. The code for this has been hidden in the next cell and does not need to be read. Make sure to run both cells. Training should take a few seconds on a GPU and around a minute on a CPU.

In [ ]:
# @title `fit`: the training loop (you don't need to read this line by line) { display-mode: "form" }
EPOCHS = 25


def fit(cnn, signals, labels, epochs=EPOCHS, lr=0.001):
    """Train a CNN; return the training and validation loss after each epoch, and the weights after each epoch."""
    cnn.to(device)
    signals, labels = signals.to(device), labels.to(device)
    optimiser = torch.optim.Adam(cnn.parameters(), lr=lr)
    rng = np.random.default_rng(SEED)
    plot = ecg.LossPlot(epochs)
    train_losses, val_losses, snapshots = [], [], []
    start = time.time()
    for epoch in range(epochs):
        cnn.train()  # training mode: BatchNorm learns from each batch
        order = rng.permutation(cnn_train_idx)  # the training ECGs in a new random order
        batch_losses = []
        for first in range(0, len(order), 32):  # 32 ECGs at a time
            rows = order[first:first + 32]
            batch_losses.append(train_step(cnn, optimiser, signals[rows], labels[rows]))
        val_scores = torch.tensor(predict(cnn, signals[val_idx]))
        train_losses.append(np.mean(batch_losses))
        val_losses.append(loss_fn(val_scores, labels[val_idx].view(-1).cpu()).item())
        snapshots.append(copy.deepcopy(cnn.state_dict()))  # a copy of every parameter's value
        plot.update(train_losses, val_losses, time.time() - start)
    return train_losses, val_losses, snapshots

In [ ]:
torch.manual_seed(SEED)
cnn = make_cnn()
train_losses, val_losses, snapshots = fit(cnn, signals, y)

### Overfitting, seen

The training loss keeps going down, but the validation loss plateaus after a few epochs and remains much higher than the training loss (it is also very noisy because only 20 of the 162 validation ECGs have AF). The next cell prints the AUROC on the training and validation ECGs after the last epoch; note the large discrepancy.

In [ ]:
for name, rows in [("training", cnn_train_idx), ("validation", val_idx)]:
    print(f"After the last epoch, AUROC on the {name} ECGs: {roc_auc_score(af[rows], predict(cnn, signals[rows].to(device))):.3f}")

The network can almost perfectly order its training ECGs; however, it fails to do the same with new patients. This is an example of overfitting. The standard solution is a process called **early stopping**, wherein you simply select the model's weights from the epoch with the lowest validation loss.

### TODO 3: keep the best epoch

`val_losses`: validation loss after each epoch. `snapshots`: weights after each epoch (in the same order as `val_losses`). Set `best` to the **position** of the lowest validation loss (not the actual loss value). `np.argmin(values)` gives the position of the smallest value in a list (this is a 0-based index, just like any Python index). Once the check is ✅, the cell will load the weights from that epoch.

In [ ]:
best = len(val_losses) - 1  # placeholder: the last epoch
if checks.best_epoch(best, val_losses, train_losses):
    cnn.load_state_dict(snapshots[best])
    print(f"Keeping epoch {best + 1}: validation loss {val_losses[best]:.3f}, training loss {train_losses[best]:.3f}")
else:
    print("The CNN keeps its weights from the last epoch until the check above shows ✅.")

### The result

We can finally use the CNN to score the test patients. The cell below prints the test AUROC for both models.

In [ ]:
x_test_signals = signals[test_idx].to(device)
cnn_auroc = roc_auc_score(af_test, predict(cnn, x_test_signals))
ecg.show_aurocs({"Two hand-made features": linear_auroc, "CNN on raw lead II": cnn_auroc})

The CNN has learned to detect AF from the raw signal about as well as the two hand-crafted features. Given only 23 AF ECGs in the test set, a few hundredths difference either way is meaningless. To truly do better, a model for ECGs needs far more than a few hundred ECGs.

**3b.** Why do we pick the best epoch with the validation patients and not the test patients?

- a) The test patients would no longer be 'unseen' and so the test score would over-estimate the performance of the model on new patients.
- b) The test set is too small.
- c) Speed of validation.
- d) It does not matter which we select.

In [ ]:
check("3b", "")

## Wrap-up: what did the network learn? (4 min)

The section of the network that creates the embedding is known as an **encoder**; in our case this is `cnn[:-1]`, the CNN with the final layer removed (`[:-1]` strips the final element, similar to a list). We can visualise the positioning of our test ECGs in the embedding by using PCA (principal component analysis), which determines the two directions of greatest variability in the 64 numbers. See if you can find where the red (AF) dots cluster together.

In [ ]:
encoder = cnn[:-1]
encoder.eval()
with torch.no_grad():
    embeddings = encoder(x_test_signals).cpu().numpy()
print("Embeddings:", embeddings.shape)
points = PCA(n_components=2).fit_transform(embeddings)
ecg.plot_embedding(points, af_test, "The CNN's embeddings of the test ECGs")

The network has, without knowing anything about what AF is (only which ECGs had the label), learnt to put most AF ECGs (red) in one region of the embedding.

### Where this leads: the Multimodal lab

This afternoon's Multimodal lab builds from this point with embeddings. We're using the exact same data file, same labels from the machine report, and the same patient train/test split. The main difference is that instead of our tiny CNN, we leverage a model called **ECGFounder**, which is a similar encoder but was trained on over 10 million ECGs. Here are both embeddings of the first AF ECG in the test set:

In [ ]:
first_af = int(np.flatnonzero(af_test)[0])  # the position of the first 1 in af_test
diagrams.handover(embeddings[first_af], int(test_idx[first_af]))

**4a.** In the Multimodal lab, every ECG has a 1024-number list called an ECGFounder embedding.
What is this?

- a) The ECG itself, downsampled to 1024 time points.
- b) The output of an encoder trained on millions of ECGs. This is a learnt summary of the ECG that can be used as features by a simple model.
- c) The 1024 most important samples from lead II.
- d) A probability of AF for each of 1024 conditions.

In [ ]:
check("4a", "")

### Take-home points

- **All neural networks are trained in the same manner:** forward, loss, backward, step.
- **Test on unseen patients**, split on a per-patient basis; the AUROC is the probability that a positive case is scored higher than a negative case.
- **Convolutional Neural Networks learn features** but are prone to overfitting on limited data; ensure the best performing epoch on validation data is kept.
- The Multimodal lab leverages a pre-trained ECG embedding network (**encoder**) trained on millions of ECGs.

## Extensions

If you have additional time, please attempt the following in any order. Please keep the same patient split, use the validation patients for model selection, and only view the test AUROC at the very end.

1. **Where does the two-feature model make errors?** Top false positives.
2. **Just how big a step?** The learning rate.
3. **A filter within PyTorch.** The custom filter with `F.conv1d`, and a smoothing filter.
4. **All 12 leads.** Does the CNN improve with all leads?
5. **Where is the CNN looking?** Saliency map of regions of the ECG that are important for the score.
6. **How trustworthy are the probabilities?** Calibration curve.
7. **Another rhythm or conduction abnormality.** Such conditions are often rarer and may impair model performance.
8. **Sequence model instead of a CNN.**

### Extension 1: where does the two-feature model go wrong?

The cell lists the five highest-scoring ECGs without AF in the test set according to the 2-feature model: the model's most confident false positive predictions. Each ECG's lead II signal is visualised with the beat detection overlaid.

In [ ]:
no_af = np.flatnonzero(af_test == 0)
false_alarms = test_idx[no_af[np.argsort(-test_scores[no_af])][:5]]  # rows of the five highest scores
for row in false_alarms:
    print(f"ECG {row}: heart rate {features['heart_rate'][row]:3.0f}, irregularity {features['rr_irregularity'][row]:.2f}. "
          f"{meta['machine_report'][row][:90]}")
ecg.plot_rhythm(ecgs, meta, false_alarms)

**Discuss:** For each ECG, look at where the red dots align with the visible beats.
Is this a case of the model being incorrect, or is it a case where the two features provided to the model are already incorrect?
Which reports are examples of true irregular rhythms in the absence of AF?

**Your answer:**


### Extension 2: how big a step? The learning rate

The learning rate is a **hyperparameter**: it is set by the person training the model, not learnt. The cell below trains the model three times from scratch for 100 steps, with learning rates of 0.01, 0.5 and 30.

In [ ]:
curves = {}
for lr in [0.01, 0.5, 30]:
    torch.manual_seed(SEED)
    trial_model = nn.Linear(2, 1)
    trial_optimiser = torch.optim.SGD(trial_model.parameters(), lr=lr)
    curves[lr] = [train_step(trial_model, trial_optimiser, x_train, y_train) for step in range(100)]
ecg.plot_learning_rates(curves)

**E2.** Take a look at the loss for a learning rate of 30.
It bounces around and ends up higher than where it started!
Why is that?

- a) The model has not trained for long enough for such a high learning rate.
- b) Each step taken is too large and overshoots the bottom of the valley and lands further up the other side.
- c) The gradients are incorrect due to the learning rate being too large.
- d) The model is too small for the learning rate.

In [ ]:
check("e2", "")

### Extension 3: a filter in PyTorch

`F.conv1d` is PyTorch's convolution operator. Note that it expects an input of the shape (ECGs, channels, time), and so `.view(1, 1, -1)` reshapes a single signal into a single ECG with a single channel. The next cell moves the hand-crafted filter from Part 3 across three seconds of the ECG.

In [ ]:
kernel = torch.tensor([-1.0, 0, 1])
signal = torch.tensor(ecgs[sinus_example, 1])
output = F.conv1d(signal.view(1, 1, -1), kernel.view(1, 1, -1)).view(-1)
ecg.plot_filter(signal.numpy(), kernel.numpy(), output.numpy())

This is maximally sensitive to the steepest point of each beat, the QRS complex. Now change `kernel` to `torch.ones(25) / 25`, which will take the average of 25 neighbouring samples, and re-run the cell to view a smoothing filter.

### Extension 4: all 12 leads

The reading of AF by a cardiologist typically occurs by looking at the rhythm strip (lead II) as well as lead V1, as it is the lead where atrial activity is best visualised. `make_cnn(in_channels=12)` implements the same network now with 12 channels inputted into the network, one for each lead. The only changes that have been made are to the first layer. Please run the cell below and compare the performance of the 12-lead CNN model to the lead II CNN model. Is the difference more than you would expect by chance given the 23 AF ECGs in the test set? Can you think of any reasons why the other leads may help or hinder performance?

In [ ]:
torch.manual_seed(SEED)
cnn12 = make_cnn(in_channels=12)
all_leads = torch.tensor(ecgs)  # (ECGs, 12 leads, 2,500 time points)
train12, val12, snapshots12 = fit(cnn12, all_leads, y)
cnn12.load_state_dict(snapshots12[int(np.argmin(val12))])
auroc12 = roc_auc_score(af_test, predict(cnn12, all_leads[test_idx].to(device)))
ecg.show_aurocs({"Two hand-made features": linear_auroc, "CNN on lead II": cnn_auroc, "CNN on 12 leads": auroc12})

### Extension 5: what does the CNN look at?

A **saliency map** tells us where the CNN is 'looking' in the ECG. It is the gradient of the CNN's score with respect to the input and so measures how much a small change in each of the 2500 samples will affect the score. This is one of the simplest explainable AI tools and one that you should be familiar with computing. Fill in the code below by replacing the placeholder with two lines. The first should compute the score of the CNN on `ecg_in` (remember that `requires_grad_()` is set to true and so PyTorch will track gradients). This score should then have `.backward()` run on it. The second line should set `saliency` to `ecg_in.grad.abs().view(-1).cpu().numpy()`. Once complete, attempt to visualise several ECGs for both AF and non-AF cases from the test set. Saliency maps tend to be noisy and so it is worthwhile to try and see where the bulk of the larger values tend to cluster. We can see if the model is looking at the QRS complexes, the spaces between the complexes, or at the baseline where the P waves (or fibrillation waves) would be.

In [ ]:
ECG_NUMBER = af_example  # try other rows, for example int(test_idx[af_test == 1][3]) or sinus_example
cnn.eval()
ecg_in = signals[ECG_NUMBER:ECG_NUMBER + 1].to(device).clone().requires_grad_()
saliency = np.zeros(ecg_in.shape[-1])  # placeholder: replace with the saliency map
checks.saliency(saliency)
ecg.plot_saliency(ecgs[ECG_NUMBER, 1], saliency, meta["machine_report"][ECG_NUMBER])

### Extension 6: are the probabilities trustworthy?

A model may be good at ordering patients (high AUROC), but the predicted probabilities may not reflect the true probabilities. A **calibration curve** is a plot of a model's predicted probabilities divided into quantiles and the true probability of the event occurring within each quantile. A perfectly calibrated model will have a calibration curve that aligns with the diagonal line. The cell below plots the calibration curve for the 2-feature model, the best epoch CNN, and the overfit last epoch CNN. Which model is best calibrated? What do you think will happen to predictions on new patients if you continue to train the model until the training loss approaches zero? How much do you trust any of these curves given that there are only 23 AF ECGs within the test set?

In [ ]:
from sklearn.calibration import calibration_curve

last_epoch_cnn = make_cnn().to(device)
last_epoch_cnn.load_state_dict(snapshots[-1])
probabilities = {"Two hand-made features": 1 / (1 + np.exp(-test_scores)),
                 "CNN, best epoch": 1 / (1 + np.exp(-predict(cnn, x_test_signals))),
                 "CNN, last epoch": 1 / (1 + np.exp(-predict(last_epoch_cnn, x_test_signals)))}
ecg.plot_calibration({name: calibration_curve(af_test, p, n_bins=5, strategy="quantile")
                      for name, p in probabilities.items()})

### Extension 7: another rhythm or conduction problem

Atrial fibrillation is relatively common (110/1000) in this dataset. Modify `PATTERN` to `"atrial flutter"` (20 ECGs), `"bundle branch block"` (72), or `"sinus bradycardia"` (113) and run the cell to repeat Parts 1-3 in one go. What conditions can the rhythm features detect, and which are the CNNs better at identifying? What happens to the test AUROC and your confidence in it if there are very few cases in your test set?

In [ ]:
PATTERN = "bundle branch block"
label = meta["machine_report"].str.contains(PATTERN, case=False).to_numpy().astype(int)
print(f"{label.sum()} ECGs, {label[test_idx].sum()} of them in the test set")
y_new = torch.tensor(label, dtype=torch.float32).unsqueeze(1)  # the new labels; y keeps the AF labels

torch.manual_seed(SEED)
model_new = nn.Linear(2, 1)
optimiser_new = torch.optim.SGD(model_new.parameters(), lr=0.5)
for step in range(100):
    train_step(model_new, optimiser_new, x_train, y_new[train_idx])
torch.manual_seed(SEED)
cnn_new = make_cnn()
train_new, val_new, snapshots_new = fit(cnn_new, signals, y_new)
cnn_new.load_state_dict(snapshots_new[int(np.argmin(val_new))])
ecg.show_aurocs({f"Two features, {PATTERN}": roc_auc_score(label[test_idx], predict(model_new, x_test)),
                 f"CNN, {PATTERN}": roc_auc_score(label[test_idx], predict(cnn_new, x_test_signals))})

### Extension 8: a sequence model instead of a CNN

**Recurrent neural networks** (RNNs) such as **LSTM** models are capable of accumulating a memory state when processing a sequence step-wise.
This may be useful for finding temporal patterns in signals.
The framework below attempts to simplify the problem for the model by first reducing the sequence length using two convolutional blocks to produce a sequence of length 156 with 32 features.
This is then fed to an LSTM model, which reads the sequence and then averages the outputs at each time step to predict a single value.
`ConvLSTM` demonstrates how this model is constructed as a class.
A class is the standard method of constructing deep learning models in PyTorch for models that are not a simple stack of layers.
`__init__` shows how the model's layers are instantiated, with `forward` demonstrating how an ECG would flow through the model.
`fit` trains the model, after which the performance of the model can be compared to the CNN by means of the AUROC on the test set as well as the loss curves and training time.
What model would you select for this task and why?

In [ ]:
class ConvLSTM(nn.Module):
    """Two convolutional blocks to shorten the signal, then an LSTM, then a linear layer."""

    def __init__(self, in_channels=1, hidden=32):
        super().__init__()
        self.shorten = nn.Sequential(conv_block(in_channels, 16), conv_block(16, 32))  # 2,500 -> 156 steps
        self.lstm = nn.LSTM(32, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        steps = self.shorten(x).transpose(1, 2)  # (ECGs, time, channels): the order an LSTM expects
        outputs, _ = self.lstm(steps)  # one output per step
        return self.head(outputs.mean(dim=1))  # average over time, then one score


torch.manual_seed(SEED)
lstm = ConvLSTM()
train_lstm, val_lstm, snapshots_lstm = fit(lstm, signals, y)
lstm.load_state_dict(snapshots_lstm[int(np.argmin(val_lstm))])
ecg.show_aurocs({"CNN on lead II": cnn_auroc,
                 "Conv + LSTM on lead II": roc_auc_score(af_test, predict(lstm, x_test_signals))})

Report on findings for different attempted extensions and the reasoning behind them.

**Your answers:**